In [1]:
from app.agent.question_fix import fix_question
from app.agent.state import LoopState
from app.agent.graph import build_graph
from app.agent.patterns import route_after_search
from langgraph.graph import END, START, StateGraph

RULE_WORDS = ("숙박비", "식비", "교통비") 
QUESTIONS = ["부산 출장 숙박비 한도가 얼마인가요?", "출장 가서 호텔비는 얼마까지 되나요?", "점심 메뉴"]


def fake_search(state):
    q = state.get("query") or state["question"]
    return {"passed": any(w in q for w in RULE_WORDS), "trace": ["search"]} 

def fake_end(name):
    def step(state):
        return {"trace": [name]}
    return step

def traced(name, step):

    def wrapped(state):
        return {**step(state), 'trace': [name]}
    return wrapped

def pick_no_limit(state):
    return 'found' if state.get('passed') else 'fix'

g = StateGraph(LoopState)
g.add_node('search', fake_search)
g.add_node('fix', traced('fix', fix_question))
g.add_node('found', fake_end('found'))
g.add_edge(START, 'search')
g.add_conditional_edges('search', pick_no_limit, {'found': 'found', 'fix': 'fix'})
g.add_edge('found', END)
g.add_edge('fix', 'search')
loop = g.compile()
out = loop.invoke({'question': QUESTIONS[1], 'user': {}})
print('trace :', out['trace'])
print('query :', out['query'])
print('fixes :', out['fixes'])


trace : ['search', 'fix', 'search', 'found']
query : 출장 가서 숙박비는 얼마까지 되나요?
fixes : 1
